# **01 - ETL & Data Cleaning**

## Objectives

- Load and inspect the raw Online Retail Transaction dataset.
- Assess data quality by identifying missing values, duplicate records, invalid quantities and unit prices.
- Clean and transform the dataset to prepare it for analysis.
- Create a `Revenue` feature to support further analysis.
- Save the cleaned dataset for exploratory data analysis and visualisation.

## Inputs

- Raw Online Retail Transaction dataset stored in the `data` folder.

## Outputs

- Cleaned Online Retail Transaction dataset.
- Cleaned data ready for exploratory data analysis and visualisation.

## Additional Comments

- Data-cleaning decisions will be based on the findings from the initial data inspection and documented throughout the notebook.

---

# Change working directory

In [1]:
import os

current_dir = os.getcwd()
print("Current directory:", current_dir)

if os.path.basename(current_dir) == "jupyter_notebooks":
    os.chdir(os.path.dirname(current_dir))
elif os.path.basename(current_dir) == "vscode-projects":
    os.chdir("Online Retail Transaction Analysis")

current_dir = os.getcwd()
print("Working directory set to:", current_dir)

Current directory: /Users/sahraosman/Documents/vscode-projects/Online Retail Transaction Analysis/jupyter_notebooks
Working directory set to: /Users/sahraosman/Documents/vscode-projects/Online Retail Transaction Analysis


---

# Section 1 -  Import libraries

Here I will be importing the libraries that I will be using in this notebook.

In [2]:
import pandas as pd


---

# Section 2 - Load the dataset

This section will load the raw Online Retail Transaction dataset and perform an initial inspection to understand its structure.

In [3]:
df = pd.read_csv("data/Online_Retail.csv")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


In [4]:
print("Number of rows in the dataset:", df.shape[0])
print("Number of columns in the dataset:", df.shape[1])

Number of rows in the dataset: 541909
Number of columns in the dataset: 8


---

# Section 3 - Initial Data Inspection

Here I'll perform an initial inspection of the dataset to understand its structure, identify any data quality issues, and determine the necessary cleaning steps.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   541909 non-null  int64  
 7   Country      541909 non-null  str    
dtypes: float64(1), int64(2), str(5)
memory usage: 33.1 MB


Looking at the structure of the dataset, we can see that it contains 8 columns and 541,909 rows. The columns are as follows:
`InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, and `Country`.

I can already see that there are some missing values in the `Description` column. I will need to investigate further to determine the extent of the missing values and decide on an appropriate strategy for handling them.

`InvoiceDate` is currently in string format, and I will need to convert it to a datetime format for analysis.

In [6]:
df.describe(include="all")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,541909,541909,540455,541909.000000,541909,541909.000000,541909.000000,541909
unique,25900,4070,4223,NaN,23260,NaN,NaN,38
top,573585,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,2011-10-31 14:41:00,NaN,NaN,United Kingdom
freq,1114,2313,2369,NaN,1114,NaN,NaN,495478
mean,NaN,NaN,NaN,9.552250,NaN,4.611114,15287.518434,NaN
std,NaN,NaN,NaN,218.081158,NaN,96.759853,1484.746041,NaN
min,NaN,NaN,NaN,-80995.000000,NaN,-11062.060000,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000,NaN,1.250000,14367.000000,NaN
50%,NaN,NaN,NaN,3.000000,NaN,2.080000,15287.000000,NaN
75%,NaN,NaN,NaN,10.000000,NaN,4.130000,16255.000000,NaN


The descriptive statistics provide an overview of both numerical and categorical variables. The dataset contains 25,900 unique invoices, 4,070 unique stock codes and transactions across 38 countries. The United Kingdom accounts for the majority of transaction records.

The `Quantity` and `UnitPrice` columns contain negative minimum values, with `Quantity` ranging from -80,995 to 80,995 and `UnitPrice` ranging from -11,062.06 to 38,970. These values require further investigation during data cleaning.

The `Description` count is lower than the total number of rows, confirming the presence of missing values. Further analysis will be performed to quantify and investigate these records.

---

# Section 4 - Missing Values

In this section, I will investigate missing values within the dataset to determine their extent and decide on an appropriate strategy for handling them.

In [7]:
df.isnull().sum()

InvoiceNo         0
StockCode         0
Description    1454
Quantity          0
InvoiceDate       0
UnitPrice         0
CustomerID        0
Country           0
dtype: int64

The `Description` column contains 1,454 missing values. I will calculate the percentage of missing values and investigate these records further before deciding how to handle them.

In [8]:
df.isnull().sum() / df.shape[0] * 100

InvoiceNo      0.000000
StockCode      0.000000
Description    0.268311
Quantity       0.000000
InvoiceDate    0.000000
UnitPrice      0.000000
CustomerID     0.000000
Country        0.000000
dtype: float64

I can see that the missing values in the `Description` column represent approximately 0.27% of the total dataset, which is a small percentage. I will need to decide whether to drop these rows or impute the missing values based on the context of the analysis.

In [9]:
df[df["Description"].isnull()].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,15287,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,15287,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,15287,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,15287,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,15287,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,15287,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,15287,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,15287,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,15287,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,15287,United Kingdom


The initial sample shows that records with a missing `Description` also have a `UnitPrice` of 0. I will investigate whether this pattern applies to all records with a missing description before deciding how to handle them.

In [10]:
df[df["Description"].isnull()]["UnitPrice"].value_counts()

UnitPrice
0.0    1454
Name: count, dtype: int64

All records with a missing `Description` also have a `UnitPrice` of 0. This confirms that the pattern identified in the initial sample applies to all 1,454 missing-description records. I will investigate the wider zero-priced records before deciding how these observations should be handled.

In [11]:
df[df["UnitPrice"] == 0].shape[0]

2515

There are 2,515 records with a `UnitPrice` of 0. Since only 1,454 of these records have a missing `Description`, there are additional zero-priced records that require further investigation.

In [12]:
df[(df["UnitPrice"] == 0) & (df["Description"].notnull())].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
6391,536941,22734,amazon,20,2010-12-03 12:08:00,0.0,15287,United Kingdom
6392,536942,22139,amazon,15,2010-12-03 12:08:00,0.0,15287,United Kingdom
7313,537032,21275,?,-30,2010-12-03 16:50:00,0.0,15287,United Kingdom
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,0.0,12647,Germany
13217,537425,84968F,check,-20,2010-12-06 15:35:00,0.0,15287,United Kingdom
13218,537426,84968E,check,-35,2010-12-06 15:36:00,0.0,15287,United Kingdom
13264,537432,35833G,damages,-43,2010-12-06 16:10:00,0.0,15287,United Kingdom
14335,537534,85064,CREAM SWEETHEART LETTER RACK,1,2010-12-07 11:48:00,0.0,15287,United Kingdom
14336,537534,84832,ZINC WILLIE WINKIE CANDLE STICK,1,2010-12-07 11:48:00,0.0,15287,United Kingdom
14337,537534,84692,BOX OF 24 COCKTAIL PARASOLS,2,2010-12-07 11:48:00,0.0,15287,United Kingdom


The additional zero-priced records contain a mixture of product descriptions and non-standard descriptions such as `check`, `damages`, and `?`. Some also contain negative quantities. These records will be investigated further during the Quantity and UnitPrice validation stage.

### Missing Value Investigation

The `Description` column contains 1,454 missing values, representing approximately 0.27% of the dataset. Further investigation showed that all 1,454 records with a missing product description also have a `UnitPrice` of 0.

Since these records do not contain a product description and have no recorded unit price, they provide limited value for the product and revenue analysis required for this project. Therefore, these records will be removed from the cleaned dataset.

Further investigation also identified additional records with a `UnitPrice` of 0 that contain valid descriptions. These records will be investigated separately during the validation of `UnitPrice` rather than being removed solely on the basis of the missing-value analysis.

In [13]:
df = df.dropna(subset=["Description"])

In [14]:
df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

After dropping the records with missing `Description` values, we can see that there are no longer any missing values in the dataset.

In [15]:
df.shape[0]

540455

Now there are 540,455 records remaining in the dataset after dropping the records with missing `Description` values. We started with 541,909 rows.

---

# Section 5 - Duplicate Records

In this section, I will investigate duplicate records within the dataset to determine their extent and decide on an appropriate strategy for handling them.

In [16]:
df.duplicated().sum()

np.int64(5268)

Pandas has found 5268 duplicate records in the dataset. I will investigate these records further to determine whether they should be removed or retained based on the context of the analysis.

In [17]:
df[df.duplicated(keep=False)].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920,United Kingdom


In [18]:
duplicates = df[df.duplicated(keep=False)]

duplicates.sort_values(by=["InvoiceNo", "StockCode", "Quantity", "UnitPrice"]).head(30)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920,United Kingdom
598,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920,United Kingdom


The duplicate inspection confirms that some records are exact copies, with identical values across all columns including `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, and `Country`.

Repeated customers, products or invoices are expected in retail transaction data and are not considered duplicates unless the entire row is identical. Exact duplicate records could cause quantities and revenue to be counted more than once, so these records will be removed before analysis.

In [19]:
df = df.drop_duplicates(keep="first")

This keeps the first occurrence of each duplicate record and removes subsequent identical occurrences.

In [20]:
df.duplicated().sum()

np.int64(0)

We can see now that after removing the duplicate records, there are no longer any duplicate records in the dataset.

In [21]:
df.shape[0]

535187

After removing 5,268 exact duplicate records, 535,187 records remain in the dataset.

### Duplicate Records Conclusion

The dataset contained 5,268 exact duplicate records. Inspection confirmed that these records had identical values across all columns.

Repeated customers, products and invoices are expected within retail transaction data and were not considered duplicates unless the entire record was identical. Exact duplicates were removed while keeping the first occurrence to prevent potential double-counting of quantities and revenue.

After removing the duplicates, no exact duplicate records remain in the dataset.

---

# Section 6 - Quantity and UnitPrice Validation

In this section, I will investigate invalid or unusual values within the `Quantity` and `UnitPrice` columns to determine which records are suitable for the sales and revenue analysis.

In [22]:
print("Quantity below 1:", (df["Quantity"] < 1).sum())

Quantity below 1: 9725


In [23]:
print("Quantity equal to 0:", (df["Quantity"] == 0).sum())

Quantity equal to 0: 0


In [24]:
print("Quantity below 0:", (df["Quantity"] < 0).sum())

Quantity below 0: 9725


There are 9,725 records where `Quantity` is below 1. Further investigation confirms that there are no zero-quantity records, meaning all 9,725 records contain negative quantities.

Negative quantities may represent returns, cancellations, damaged stock or other adjustments rather than completed sales. I will inspect these records further before deciding how they should be handled for the sales and demand analysis.

In [25]:
df[df["Quantity"] < 0].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897,United Kingdom


In [26]:
negative_quantity = df[df["Quantity"] < 0]

negative_quantity["InvoiceNo"].str.startswith("C").value_counts()

InvoiceNo
True     9251
False     474
Name: count, dtype: int64

In [27]:
non_c_negative = negative_quantity[~negative_quantity["InvoiceNo"].str.startswith("C")]

non_c_negative.head(30)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
7313,537032,21275,?,-30,2010-12-03 16:50:00,0.0,15287,United Kingdom
13217,537425,84968F,check,-20,2010-12-06 15:35:00,0.0,15287,United Kingdom
13218,537426,84968E,check,-35,2010-12-06 15:36:00,0.0,15287,United Kingdom
13264,537432,35833G,damages,-43,2010-12-06 16:10:00,0.0,15287,United Kingdom
21338,538072,22423,faulty,-13,2010-12-09 14:10:00,0.0,15287,United Kingdom
21518,538090,20956,?,-723,2010-12-09 14:48:00,0.0,15287,United Kingdom
22296,538161,46000S,Dotcom sales,-100,2010-12-09 17:25:00,0.0,15287,United Kingdom
22297,538162,46000M,Dotcom sales,-100,2010-12-09 17:25:00,0.0,15287,United Kingdom
42564,540010,22501,reverse 21/5/10 adjustment,-100,2011-01-04 11:13:00,0.0,15287,United Kingdom
42566,540012,22502,reverse 21/5/10 adjustment,-100,2011-01-04 11:14:00,0.0,15287,United Kingdom


In [28]:
print("Non-C negative records:", non_c_negative.shape[0])
print("With UnitPrice equal to 0:", (non_c_negative["UnitPrice"] == 0).sum())

Non-C negative records: 474
With UnitPrice equal to 0: 474


### Negative Quantity Investigation

There are 9,725 records with a negative `Quantity` and no records where `Quantity` is equal to 0.

Further investigation showed that 9,251 of the negative-quantity records have an `InvoiceNo` beginning with `C`. These records contain negative quantities associated with products and are consistent with cancelled or returned transactions.

The remaining 474 negative-quantity records do not have an invoice number beginning with `C`. Inspection identified descriptions such as `damages`, `faulty`, `thrown away`, `Given away`, `samples/damages`, and stock adjustments. Further validation confirmed that all 474 of these records also have a `UnitPrice` of 0, providing additional evidence that they do not represent ordinary revenue-generating customer sales.

As this project focuses on revenue, quantity sold and customer demand, records with negative quantities will be excluded from the analysis because they do not represent completed positive sales.

In [29]:
df = df[df["Quantity"] >= 1]

In [30]:
(df["Quantity"] < 1).sum()

np.int64(0)

In [31]:
df.shape[0]

525462

After removing the 9,725 records with negative quantities, 525,462 records remain in the dataset. A validation check confirms that there are now no records where `Quantity` is below 1.

In [32]:
print("Unit Price below 0:", (df["UnitPrice"] < 0).sum())

Unit Price below 0: 2


In [33]:
print("Unit Price equal to 0:", (df["UnitPrice"] == 0).sum())

Unit Price equal to 0: 582


In [34]:
print("Unit Price above 0:", (df["UnitPrice"] > 0).sum())

Unit Price above 0: 524878


There are 2 records with a negative `UnitPrice`, 582 records with a `UnitPrice` of 0, and 524,878 records with a positive `UnitPrice`. Since zero and negative prices may represent adjustments, errors or non-standard transactions rather than completed sales, I will investigate these records further before deciding how they should be handled.

In [35]:
df[df["UnitPrice"] < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,15287,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,15287,United Kingdom


The two records with a negative `UnitPrice` were inspected and both have the description `Adjust bad debt`, a stock code of `B`, and invoice numbers beginning with `A`. These records appear to represent accounting adjustments rather than product sales. As this analysis focuses on completed retail sales and revenue, these records will be excluded from the cleaned dataset.

In [36]:
df[df["UnitPrice"] == 0].head(30)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
6391,536941,22734,amazon,20,2010-12-03 12:08:00,0.0,15287,United Kingdom
6392,536942,22139,amazon,15,2010-12-03 12:08:00,0.0,15287,United Kingdom
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,0.0,12647,Germany
14335,537534,85064,CREAM SWEETHEART LETTER RACK,1,2010-12-07 11:48:00,0.0,15287,United Kingdom
14336,537534,84832,ZINC WILLIE WINKIE CANDLE STICK,1,2010-12-07 11:48:00,0.0,15287,United Kingdom
14337,537534,84692,BOX OF 24 COCKTAIL PARASOLS,2,2010-12-07 11:48:00,0.0,15287,United Kingdom
14338,537534,48184,DOORMAT ENGLISH ROSE,3,2010-12-07 11:48:00,0.0,15287,United Kingdom
14339,537534,48111,DOORMAT 3 SMILEY CATS,1,2010-12-07 11:48:00,0.0,15287,United Kingdom
14340,537534,22697,GREEN REGENCY TEACUP AND SAUCER,1,2010-12-07 11:48:00,0.0,15287,United Kingdom
14341,537534,22682,FRENCH BLUE METAL DOOR SIGN 7,1,2010-12-07 11:48:00,0.0,15287,United Kingdom


In [37]:
df[df["UnitPrice"] == 0]["CustomerID"].value_counts().head(20)

CustomerID
15287    542
13081      4
14646      4
14911      2
12415      2
13985      2
12647      1
16560      1
15107      1
17560      1
13239      1
13113      1
14410      1
12457      1
17667      1
16818      1
12507      1
15581      1
16133      1
12748      1
Name: count, dtype: int64

In [38]:
df[df["UnitPrice"] == 0]["InvoiceNo"].nunique()

227

In [39]:
zero_price_other_customers = df[(df["UnitPrice"] == 0) & (df["CustomerID"] != 15287)]

zero_price_other_customers

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,0.0,12647,Germany
33576,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,2010-12-16 14:36:00,0.0,16560,United Kingdom
40089,539722,22423,REGENCY CAKESTAND 3 TIER,10,2010-12-21 13:45:00,0.0,14911,EIRE
47068,540372,22090,PAPER BUNTING RETROSPOT,24,2011-01-06 16:41:00,0.0,13081,United Kingdom
47070,540372,22553,PLASTERS IN TIN SKULLS,24,2011-01-06 16:41:00,0.0,13081,United Kingdom
56674,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,2011-01-13 15:10:00,0.0,15107,United Kingdom
86789,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,2011-02-10 13:08:00,0.0,17560,United Kingdom
130188,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,2011-03-23 10:25:00,0.0,13239,United Kingdom
139453,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,2011-03-30 12:45:00,0.0,13113,United Kingdom
145208,548871,22162,HEART GARLAND RUSTIC PADDED,2,2011-04-04 14:42:00,0.0,14410,United Kingdom


## Zero and Negative Unit Price Investigation
There are 582 records with a `UnitPrice` of 0 and 2 records with a negative `UnitPrice`. The two negative-price records are labelled `Adjust bad debt`, indicating financial adjustments rather than product sales.

Inspection of the zero-priced records identified a mixture of normal product descriptions and non-standard records such as `Manual`. As the reason for the zero prices cannot be reliably determined from the dataset, these records cannot be treated as confirmed revenue-generating sales. Including their quantities could also distort measures of product demand despite contributing no revenue.

Therefore, as this project focuses on completed sales, revenue and customer demand, records where `UnitPrice` is less than or equal to 0 will be excluded from the analysis.

In [40]:
df = df[df["UnitPrice"] > 0]

In [41]:
print("Unit Price equal to or less than 0:", (df["UnitPrice"] <= 0).sum())

Unit Price equal to or less than 0: 0


In [42]:
df.shape[0]

524878

After removing the 584 records with a zero or negative `UnitPrice`, 524,878 records remain. A validation check confirms that all remaining records have a positive `UnitPrice`.

---

# Section 7 - Data Type Conversion

In this section, I will review the data types within the cleaned dataset and convert any columns that require a more appropriate data type for analysis.

In [43]:
df.dtypes

InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID       int64
Country            str
dtype: object

The data type inspection shows that most columns already have suitable data types for the analysis. However, `InvoiceDate` is currently stored as a string. Since this column represents dates and times and will be used for time-based analysis, it will be converted to a datetime data type.

In [44]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], format="%Y-%m-%d %H:%M:%S")

In [45]:
df.dtypes

InvoiceNo                 str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID              int64
Country                   str
dtype: object

The `InvoiceDate` column was converted from a string to a datetime data type. This will allow the transaction dates to be used for time-based analysis, such as analysing monthly revenue trends.

In [46]:
df["Country"].nunique()

38

In [47]:
df["Country"].value_counts()

Country
United Kingdom          479985
Germany                   9025
France                    8392
EIRE                      7879
Spain                     2479
Netherlands               2359
Belgium                   2031
Switzerland               1958
Portugal                  1492
Australia                 1181
Norway                    1071
Italy                      758
Channel Islands            747
Finland                    685
Cyprus                     603
Sweden                     450
Unspecified                442
Austria                    398
Denmark                    380
Poland                     330
Japan                      321
Israel                     292
Hong Kong                  280
Singapore                  222
Iceland                    182
USA                        179
Canada                     151
Greece                     145
Malta                      112
United Arab Emirates        68
European Community          60
RSA                         57


The `Country` column contains 38 unique values. Two non-country-specific categories were identified: `Unspecified` and `European Community`. I will investigate the Unspecified records further before deciding how they should be handled.

In [48]:
df[df["Country"] == "Unspecified"].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
152712,549687,20685,DOORMAT RED RETROSPOT,2,2011-04-11 13:29:00,7.95,12363,Unspecified
152713,549687,22691,DOORMAT WELCOME SUNRISE,2,2011-04-11 13:29:00,7.95,12363,Unspecified
152714,549687,48116,DOORMAT MULTICOLOUR STRIPE,2,2011-04-11 13:29:00,7.95,12363,Unspecified
152715,549687,21213,PACK OF 72 SKULL CAKE CASES,24,2011-04-11 13:29:00,0.55,12363,Unspecified
152716,549687,21977,PACK OF 60 PINK PAISLEY CAKE CASES,24,2011-04-11 13:29:00,0.55,12363,Unspecified
152717,549687,21976,PACK OF 60 MUSHROOM CAKE CASES,24,2011-04-11 13:29:00,0.55,12363,Unspecified
152718,549687,21212,PACK OF 72 RETROSPOT CAKE CASES,24,2011-04-11 13:29:00,0.55,12363,Unspecified
152719,549687,84992,72 SWEETHEART FAIRY CAKE CASES,24,2011-04-11 13:29:00,0.55,12363,Unspecified
152720,549687,84991,60 TEATIME FAIRY CAKE CASES,24,2011-04-11 13:29:00,0.55,12363,Unspecified
152721,549687,21974,SET OF 36 PAISLEY FLOWER DOILIES,12,2011-04-11 13:29:00,1.45,12363,Unspecified


The `Country` column contains 442 records labelled as `Unspecified`. Inspection showed valid product descriptions, positive quantities and positive unit prices, indicating that these represent valid sales transactions where the geographic location is unknown. Therefore, these records will be retained. For country-level analysis, `Unspecified` will be treated separately rather than assigned to a specific country.

`European Community` will also be retained to preserve the original transaction data, but as it is not an individual country, it may be excluded from analyses specifically comparing countries.

---

# Section 8 - Feature Engineering: Revenue

In this section, I will be creating a new feature called `Revenue` by multiplying the `Quantity` and `UnitPrice` columns. This will allow for analysis of total revenue generated by each transaction.

In [49]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

In [50]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [51]:
df["Revenue"].dtype

dtype('float64')

In [52]:
df[["Quantity", "UnitPrice", "Revenue"]].head(10)

,Quantity,UnitPrice,Revenue
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34
5,2,7.65,15.30
6,6,4.25,25.50
7,6,1.85,11.10
8,6,1.85,11.10
9,32,1.69,54.08


A new `Revenue` feature was created by multiplying `Quantity` by `UnitPrice` for each transaction. The resulting values were checked against a sample of records to confirm that the calculation was correct. This feature will be used in the exploratory data analysis to investigate revenue across customers, products, countries and time periods.

# Section 9 - Final Validation and Save Cleaned Dataset

In this section, I will perform final validation checks on the cleaned dataset to confirm that the cleaning and transformation steps have been applied successfully. I will then export the cleaned dataset for use in exploratory data analysis, visualisation and later machine learning analysis.

In [53]:
print("Final number of rows:", df.shape[0])

Final number of rows: 524878


In [54]:
print("Final number of columns:", df.shape[1])

Final number of columns: 9


The final cleaned dataset contains 524,878 records and 9 columns, including the newly created `Revenue` feature.

In [55]:
print("Missing Values:", df.isnull().sum().sum())

Missing Values: 0


In [56]:
print("Duplicate Records:", df.duplicated().sum())

Duplicate Records: 0


In [57]:
print("Quantity below 1:", (df["Quantity"] < 1).sum())

Quantity below 1: 0


In [58]:
print("Unit Price equal to or below 0:", (df["UnitPrice"] <= 0).sum())

Unit Price equal to or below 0: 0


The final validation checks confirm that there are no remaining missing values or duplicate records in the dataset. All `Quantity` values are 1 or greater and all `UnitPrice` values are greater than 0, confirming that the cleaning steps were applied successfully.

In [59]:
df.info()

<class 'pandas.DataFrame'>
Index: 524878 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    524878 non-null  str           
 1   StockCode    524878 non-null  str           
 2   Description  524878 non-null  str           
 3   Quantity     524878 non-null  int64         
 4   InvoiceDate  524878 non-null  datetime64[us]
 5   UnitPrice    524878 non-null  float64       
 6   CustomerID   524878 non-null  int64         
 7   Country      524878 non-null  str           
 8   Revenue      524878 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(4)
memory usage: 40.0 MB


### Export Cleaned Dataset

Following the final validation checks, the cleaned dataset will be exported as a CSV file so that it can be loaded directly into the subsequent analysis notebooks without repeating the data-cleaning process.

In [60]:
df.to_csv("outputs/Online_Retail_Cleaned.csv", index=False)

In [61]:
os.path.exists("outputs/Online_Retail_Cleaned.csv")

True

The cleaned dataset was successfully exported to the `outputs` folder as `Online_Retail_Cleaned.csv`. The file was verified to confirm that it was created successfully and is ready to be loaded into the subsequent stages of the project.

# Conclusions & Next Steps

The ETL and data-cleaning process prepared the Online Retail Transaction dataset for further analysis. The raw dataset initially contained 541,909 records and was assessed for missing values, duplicate records, invalid quantities, invalid unit prices and inappropriate data types.

Missing product descriptions, exact duplicate records, negative quantities and zero or negative unit prices were investigated and removed where they did not represent valid completed sales. The `InvoiceDate` column was converted to a datetime data type, while valid transactions with `Unspecified` geographic information were retained.

A new `Revenue` feature was created by multiplying `Quantity` by `UnitPrice` to support revenue-based analysis. Following the cleaning and transformation process, the final dataset contains 524,878 records and 9 columns, with no remaining missing values, exact duplicates, quantities below 1 or non-positive unit prices.

The cleaned dataset was exported as `Online_Retail_Cleaned.csv` and will be used in the next stage of the project for exploratory data analysis and visualisation. Further analysis will investigate revenue trends, sales volume, customer purchasing behaviour, product performance and geographic patterns. The findings from the exploratory analysis will also help inform the subsequent machine learning stage of the project.